# Author and deploy an agent with short-term memory using Databricks Lakebase


This notebook demonstrates how to build an agent with short-term memory using the Agent Framework and Lakebase as the agent’s durable memory and checkpoint store. 

Threads allow you to store conversational state in Lakebase so you can pass in thread IDs into your agent instead of needing to send the full conversation history.

In this notebook, you will:
1. Author an agent graph with Lakebase to manage state using thread ids in a Databricks Agent 
2. Wrap the LangGraph agent with `ResponsesAgent` interface to ensure compatibility with Databricks features
3. Test the agent's behavior locally
4. Register model to Unity Catalog, log and deploy the agent for use in Review App, Playground, etc

## Prerequisites
- A Lakebase instance ready and running, see documentation ([AWS](https://docs.databricks.com/aws/en/oltp/create/) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/oltp/create/)). 
  - You can create a Lakebase instance by going to SQL Warehouses -> Lakebase Postgres -> Create database instance. You will need to retrieve values from the "Connection details" section of your Lakebase to fill out this notebook.
- Complete all the "TODO"s throughout this notebook

### Install dependencies

In [0]:
%pip install -U -qqqq databricks-langchain[memory] uv databricks-agents mlflow-skinny[databricks]
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## First time setup only: Set up checkpoint tables with your Lakebase instance

In [0]:
# First-time checkpoint table setup
from databricks.sdk import WorkspaceClient
from databricks_langchain import CheckpointSaver

# --- TODO: Fill in Lakebase project name and branch ---
PROJECT_NAME = "simple-lakebase-centric-workshop"
BRANCH = "production"  # default branch for autoscaling instances

# Create tables if missing
with CheckpointSaver(project=PROJECT_NAME, branch=BRANCH) as saver:
    saver.setup() # sets up checkpoint tables
    print("Checkpoint tables are ready.")

Checkpoint tables are ready.


# Define the agent in code

## Write agent code to file agent.py
Define the agent code in a single cell below. This lets you write the agent code to a local Python file, using the `%%writefile` magic command, for subsequent logging and deployment.

## Wrap the LangGraph agent using the ResponsesAgent interface
For compatibility with Databricks AI features, the `LangGraphResponsesAgent` class implements the `ResponsesAgent` interface to wrap the LangGraph agent.

Databricks recommends using `ResponsesAgent` as it simplifies authoring multi-turn conversational agents using an open source standard. See MLflow's [ResponsesAgent documentation](https://www.mlflow.org/docs/latest/llms/responses-agent-intro/).

In [0]:
%%writefile agent.py
import logging
import os
import uuid
from typing import Annotated, Any, Generator, Optional, Sequence, TypedDict

import mlflow
from databricks_langchain import (
    ChatDatabricks,
    UCFunctionToolkit,
    CheckpointSaver,
)
from databricks.sdk import WorkspaceClient
from langchain_core.messages import AIMessage, AIMessageChunk, AnyMessage
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
)

logger = logging.getLogger(__name__)
logging.basicConfig(level=os.getenv("LOG_LEVEL", "INFO"))

############################################
# Define your LLM endpoint and system prompt
############################################
# TODO: Replace with your model serving endpoint
LLM_ENDPOINT_NAME = "databricks-claude-opus-4-6"

# TODO: Update with your system prompt
SYSTEM_PROMPT = """You are an internal support assistant for the data engineering team. You help team members with questions about data pipelines, table schemas, job failures, and platform best practices.

Key guidelines:
- Remember context from earlier in the conversation to avoid asking the user to repeat themselves.
- When discussing tables, always use fully qualified names (catalog.schema.table).
- If a user reports a pipeline failure, ask for the job run ID and error message before troubleshooting.
- Provide concise, actionable answers. Link to relevant documentation when appropriate.
- If you don't know the answer, say so clearly rather than guessing."""

############################################
# Lakebase Autoscaling configuration
############################################
# TODO: Fill in Lakebase project name and branch
LAKEBASE_PROJECT_NAME = "simple-lakebase-centric-workshop"
LAKEBASE_BRANCH = "production"

###############################################################################
## Define tools for your agent,enabling it to retrieve data or take actions
## beyond text generation
## To create and see usage examples of more tools, see
## https://docs.databricks.com/en/generative-ai/agent-framework/agent-tool.html
###############################################################################
tools = []

# Example UC tools; add your own as needed
UC_TOOL_NAMES: list[str] = []
if UC_TOOL_NAMES:
    uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
    tools.extend(uc_toolkit.tools)

# Use Databricks vector search indexes as tools
# See https://docs.databricks.com/en/generative-ai/agent-framework/unstructured-retrieval-tools.html#locally-develop-vector-search-retriever-tools-with-ai-bridge
# List to store vector search tool instances for unstructured retrieval.
VECTOR_SEARCH_TOOLS = []

# To add vector search retriever tools,
# use VectorSearchRetrieverTool and create_tool_info,
# then append the result to TOOL_INFOS.
# Example:
# VECTOR_SEARCH_TOOLS.append(
#     VectorSearchRetrieverTool(
#         index_name="",
#         # filters="..."
#     )
# )

tools.extend(VECTOR_SEARCH_TOOLS)

#####################
## Define agent logic
#####################

class AgentState(TypedDict):
    messages: Annotated[Sequence[AnyMessage], add_messages]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]

class LangGraphResponsesAgent(ResponsesAgent):
    """Stateful agent using ResponsesAgent with pooled Lakebase Autoscaling checkpointing."""

    def __init__(self, project_name: str, branch: str):
        self.workspace_client = WorkspaceClient()
        self.project_name = project_name
        self.branch = branch

        self.model = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
        self.system_prompt = SYSTEM_PROMPT
        self.model_with_tools = self.model.bind_tools(tools) if tools else self.model

    def _create_graph(self, checkpointer: Any):
        def should_continue(state: AgentState):
            messages = state["messages"]
            last_message = messages[-1]
            if isinstance(last_message, AIMessage) and last_message.tool_calls:
                return "continue"
            return "end"

        preprocessor = (
            RunnableLambda(lambda state: [{"role": "system", "content": self.system_prompt}] + state["messages"])
            if self.system_prompt
            else RunnableLambda(lambda state: state["messages"])
        )
        model_runnable = preprocessor | self.model_with_tools

        def call_model(state: AgentState, config: RunnableConfig):
            response = model_runnable.invoke(state, config)
            return {"messages": [response]}

        workflow = StateGraph(AgentState)
        workflow.add_node("agent", RunnableLambda(call_model))

        if tools:
            workflow.add_node("tools", ToolNode(tools))
            workflow.add_conditional_edges("agent", should_continue, {"continue": "tools", "end": END})
            workflow.add_edge("tools", "agent")
        else:
            workflow.add_edge("agent", END)

        workflow.set_entry_point("agent")
        return workflow.compile(checkpointer=checkpointer)

    def _get_or_create_thread_id(self, request: ResponsesAgentRequest) -> str:
        """Get thread_id from request or create a new one.

        Priority:
        1. Use thread_id from custom_inputs if present
        2. Use conversation_id from chat context if available
        3. Generate a new UUID

        Returns:
            thread_id: The thread identifier to use for this conversation
        """
        ci = dict(request.custom_inputs or {})

        if "thread_id" in ci:
            return ci["thread_id"]

        # using conversation id from chat context as thread id
        # https://mlflow.org/docs/latest/api_reference/python_api/mlflow.types.html#mlflow.types.agent.ChatContext
        if request.context and getattr(request.context, "conversation_id", None):
            return request.context.conversation_id

        # Generate new thread_id
        return str(uuid.uuid4())

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        thread_id = self._get_or_create_thread_id(request)
        ci = dict(request.custom_inputs or {})
        ci["thread_id"] = thread_id
        request.custom_inputs = ci

        # Convert incoming Responses messages to ChatCompletions format
        # LangChain will automatically convert from ChatCompletions to LangChain format
        cc_msgs = self.prep_msgs_for_cc_llm([i.model_dump() for i in request.input])
        langchain_msgs = cc_msgs
        checkpoint_config = {"configurable": {"thread_id": thread_id}}

        with CheckpointSaver(project=self.project_name, branch=self.branch) as checkpointer:
            graph = self._create_graph(checkpointer)

            for event in graph.stream(
                {"messages": langchain_msgs},
                checkpoint_config,
                stream_mode=["updates", "messages"],
            ):
                if event[0] == "updates":
                    for node_data in event[1].values():
                        if len(node_data.get("messages", [])) > 0:
                            yield from output_to_responses_items_stream(node_data["messages"])
                elif event[0] == "messages":
                    try:
                        chunk = event[1][0]
                        if isinstance(chunk, AIMessageChunk) and chunk.content:
                            yield ResponsesAgentStreamEvent(
                                **self.create_text_delta(delta=chunk.content, item_id=chunk.id),
                            )
                    except Exception as exc:
                        logger.error("Error streaming chunk: %s", exc)


# ----- Export model -----
mlflow.langchain.autolog()
AGENT = LangGraphResponsesAgent(LAKEBASE_PROJECT_NAME, LAKEBASE_BRANCH)
mlflow.models.set_model(AGENT)

Writing agent.py


# Test the Agent locally

In [0]:
dbutils.library.restartPython()

In [0]:
from agent import AGENT
# Message 1, don't include thread_id (creates new thread)
result = AGENT.predict({
    "input": [{"role": "user", "content": "I am working on stateful agents"}]
})
print(result.model_dump(exclude_none=True))
thread_id = result.custom_outputs["thread_id"]

{'object': 'response', 'output': [{'type': 'message', 'id': 'lc_run--019e3df9-8bc8-7ed0-8a9e-9a3e101cb9da', 'content': [{'text': "That's an interesting topic! Stateful agents are a powerful pattern in AI/software engineering. Could you give me a bit more context on what you're working on? For example:\n\n- **Are you building stateful agents as part of a data pipeline?** (e.g., agents that maintain state across streaming micro-batches or orchestration steps)\n- **Are you working on AI/LLM-based agents** that need to persist memory/context across interactions?\n- **Is this related to a specific platform or framework?** (e.g., LangGraph, Databricks, Apache Flink, custom orchestration)\n\nHappy to help with architecture patterns, state management strategies, persistence layers, or troubleshooting — just let me know where you'd like to dive in!", 'type': 'output_text', 'annotations': []}], 'role': 'assistant'}], 'custom_outputs': {'thread_id': '0a1e3833-ae8a-4c7a-bdd2-fb72ce08b4a9'}}


Trace(trace_id=tr-94eaf643be300b3a8eae061fb3e8b6b4)

In [0]:
# Message 2, include thread ID and notice how agent remembers context from previous predict message
response2 = AGENT.predict({
    "input": [{"role": "user", "content": "What am I working on?"}],
    "custom_inputs": {"thread_id": thread_id}
})
print("Response 2:", response2.model_dump(exclude_none=True))

Response 2: {'object': 'response', 'output': [{'type': 'message', 'id': 'lc_run--019e3dfa-3e15-7672-b904-0e7aa6f0ba14', 'content': [{'text': "You're working on **stateful agents** — that's what you mentioned at the start of our conversation. You haven't yet shared more specifics about the context (e.g., framework, use case, or how it relates to your data engineering work), so just let me know how I can help!", 'type': 'output_text', 'annotations': []}], 'role': 'assistant'}], 'custom_outputs': {'thread_id': '0a1e3833-ae8a-4c7a-bdd2-fb72ce08b4a9'}}


Trace(trace_id=tr-9e44479246e0a87fa37d51ff5fd82ab9)

In [0]:
# Example calling agent without passing in thread id - notice it does not retain the memory
response3 = AGENT.predict({
    "input": [{"role": "user", "content": "What am I working on?"}],
})
print("Response 3 No thread id passed:", response3.model_dump(exclude_none=True))

Response 3 No thread id passed: {'object': 'response', 'output': [{'type': 'message', 'id': 'lc_run--019e3dfa-7b6c-73c1-ad70-fc494fd7a7e6', 'content': [{'text': "I don't have any context about what you're currently working on — this appears to be the start of our conversation. Could you fill me in? For example:\n\n- Are you troubleshooting a pipeline failure?\n- Working on a new table or schema design?\n- Debugging a job or query?\n- Something else entirely?\n\nHappy to help once I know what you're dealing with!", 'type': 'output_text', 'annotations': []}], 'role': 'assistant'}], 'custom_outputs': {'thread_id': '5fb03073-eb88-476b-9693-34c5ae40633c'}}


Trace(trace_id=tr-805fb00831668fc4e34568683a8c1479)

In [0]:
# predict stream example
for chunk in AGENT.predict_stream({
    "input": [{"role": "user", "content": "What am I working on?"}],
    "custom_inputs": {"thread_id": thread_id}
}):
    print("Chunk:", chunk.model_dump(exclude_none=True))

Chunk: {'type': 'response.output_text.delta', 'item_id': 'lc_run--019e3dfa-bf8d-7611-9ab4-4c8f9d873bcc', 'delta': 'You'}
Chunk: {'type': 'response.output_text.delta', 'item_id': 'lc_run--019e3dfa-bf8d-7611-9ab4-4c8f9d873bcc', 'delta': "'re working on **stateful agents**"}
Chunk: {'type': 'response.output_text.delta', 'item_id': 'lc_run--019e3dfa-bf8d-7611-9ab4-4c8f9d873bcc', 'delta': '.'}
Chunk: {'type': 'response.output_text.delta', 'item_id': 'lc_run--019e3dfa-bf8d-7611-9ab4-4c8f9d873bcc', 'delta': ' That'}
Chunk: {'type': 'response.output_text.delta', 'item_id': 'lc_run--019e3dfa-bf8d-7611-9ab4-4c8f9d873bcc', 'delta': "'s all"}
Chunk: {'type': 'response.output_text.delta', 'item_id': 'lc_run--019e3dfa-bf8d-7611-9ab4-4c8f9d873bcc', 'delta': ' the'}
Chunk: {'type': 'response.output_text.delta', 'item_id': 'lc_run--019e3dfa-bf8d-7611-9ab4-4c8f9d873bcc', 'delta': ' context I have so'}
Chunk: {'type': 'response.output_text.delta', 'item_id': 'lc_run--019e3dfa-bf8d-7611-9ab4-4c8f9d873bcc'

Trace(trace_id=tr-7fbccc9149013411498b0992c8edb017)

In [0]:
# example using conversation_id from ChatContext as thread_id
# https://mlflow.org/docs/latest/api_reference/python_api/mlflow.types.html#mlflow.types.agent.ChatContext
from agent import AGENT
import mlflow
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ChatContext
)

conversation_id = "e12345-b237-484f-ad6e-f000551703f5"

req = ResponsesAgentRequest(
    input=[{"role": "user", "content": "I am working on stateful agents, and I love the beaches, specially white sand beaches. I don't like mountains"}],
    context=ChatContext(
        conversation_id=conversation_id,
        user_id="ananyaroy@databricks.com"
    )
)
result = AGENT.predict(req)

print(result.model_dump(exclude_none=True))
thread_id = result.custom_outputs["thread_id"]
print(f"Resolved thread_id from agent: {thread_id}")

{'object': 'response', 'output': [{'type': 'message', 'id': 'lc_run--019e3dfc-cad3-7623-9e0c-682cce31b811', 'content': [{'text': "Noted! You're working on **stateful agents**.\n\nAs for the beaches and mountains — I appreciate you sharing, but as a data engineering support assistant, I'm best suited to help with things like pipelines, schemas, platform questions, and technical challenges. 😄\n\nIs there something specific about your stateful agents work I can help with? For example:\n\n- State persistence and storage patterns\n- Integration with your data pipelines or platform\n- Troubleshooting a failure or issue\n- Architecture or design questions\n\nLet me know how I can be useful!", 'type': 'output_text', 'annotations': []}], 'role': 'assistant'}], 'custom_outputs': {'thread_id': 'e12345-b237-484f-ad6e-f000551703f5'}}
Resolved thread_id from agent: e12345-b237-484f-ad6e-f000551703f5


Trace(trace_id=tr-e3677e5245bac84d4e2b97ea58a7f4bf)

In [0]:
# predict stream example
for chunk in AGENT.predict_stream({
    "input": [{"role": "user", "content": "What am I working on , and is there any personal preference i have shared with you earlier?"}],
    "custom_inputs": {"thread_id": conversation_id}
}):
    print("Chunk:", chunk.model_dump(exclude_none=True))

Chunk: {'type': 'response.output_text.delta', 'item_id': 'lc_run--019e3dfd-77d8-7ef3-ad10-2e67a0a1e5bc', 'delta': 'Here'}
Chunk: {'type': 'response.output_text.delta', 'item_id': 'lc_run--019e3dfd-77d8-7ef3-ad10-2e67a0a1e5bc', 'delta': "'s what I"}
Chunk: {'type': 'response.output_text.delta', 'item_id': 'lc_run--019e3dfd-77d8-7ef3-ad10-2e67a0a1e5bc', 'delta': ' know'}
Chunk: {'type': 'response.output_text.delta', 'item_id': 'lc_run--019e3dfd-77d8-7ef3-ad10-2e67a0a1e5bc', 'delta': ' from'}
Chunk: {'type': 'response.output_text.delta', 'item_id': 'lc_run--019e3dfd-77d8-7ef3-ad10-2e67a0a1e5bc', 'delta': ' our conversation:\n\n-'}
Chunk: {'type': 'response.output_text.delta', 'item_id': 'lc_run--019e3dfd-77d8-7ef3-ad10-2e67a0a1e5bc', 'delta': ' **You'}
Chunk: {'type': 'response.output_text.delta', 'item_id': 'lc_run--019e3dfd-77d8-7ef3-ad10-2e67a0a1e5bc', 'delta': "'re working on:**"}
Chunk: {'type': 'response.output_text.delta', 'item_id': 'lc_run--019e3dfd-77d8-7ef3-ad10-2e67a0a1e5bc', 

Trace(trace_id=tr-5d14d572bf031a85167279c833ccf5bf)

# Log the agent as an MLflow model
Log the agent as code from the agent.py file. See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

## Enable automatic authentication for Databricks resources
For the most common Databricks resource types, Databricks supports and recommends declaring resource dependencies for the agent upfront during logging. This enables automatic authentication passthrough when you deploy the agent. With automatic authentication passthrough, Databricks automatically provisions, rotates, and manages short-lived credentials to securely access these resource dependencies from within the agent endpoint.

To enable automatic authentication, specify the dependent Databricks resources when calling `mlflow.pyfunc.log_model()`.

**TODO:** 
- Add lakebase as a resource type
- If your Unity Catalog tool queries a [vector search index](https://docs.databricks.com/docs%20link) or leverages [external functions](https://docs.databricks.com/docs%20link), you need to include the dependent vector search index and UC connection objects, respectively, as resources. See docs ([AWS](https://docs.databricks.com/generative-ai/agent-framework/log-agent.html#specify-resources-for-automatic-authentication-passthrough) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-framework/log-agent#resources)).

In [0]:
# Determine Databricks resources to specify for automatic auth passthrough at deployment time
import mlflow
from agent import tools, LLM_ENDPOINT_NAME, LAKEBASE_PROJECT_NAME, LAKEBASE_BRANCH
from databricks_langchain import VectorSearchRetrieverTool
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint, DatabricksLakebase
from unitycatalog.ai.langchain.toolkit import UnityCatalogTool
from pkg_resources import get_distribution

resources = [
    DatabricksServingEndpoint(LLM_ENDPOINT_NAME),
    DatabricksLakebase(database_instance_name=LAKEBASE_PROJECT_NAME),
]

for tool in tools:
    if isinstance(tool, VectorSearchRetrieverTool):
        resources.extend(tool.resources)
    elif isinstance(tool, UnityCatalogTool):
        resources.append(DatabricksFunction(function_name=tool.uc_function_name))

input_example = {
    "input": [
        {
            "role": "user",
            "content": "What is an LLM agent?"
        }
    ],
    "custom_inputs": {"thread_id": "test-thread"},
}

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        input_example=input_example,
        resources=resources,
        pip_requirements=[
            f"databricks-langchain[memory]=={get_distribution('databricks-langchain').version}",
            f"langgraph=={get_distribution('langgraph').version}",
        ]
    )

🔗 View Logged Model at: https://adb-984752964297111.11.azuredatabricks.net/ml/experiments/519428779699050/models/m-494e46c7e88747ab8d2eaabdc75a5b12?o=984752964297111
2026/05/19 02:17:20 INFO mlflow.pyfunc: Predicting on input example to validate output


# Evaluate the agent with Agent Evaluation
Use Agent Evaluation to evalaute the agent's responses based on expected responses and other evaluation criteria. Use the evaluation criteria you specify to guide iterations, using MLflow to track the computed quality metrics. See Databricks documentation ([AWS](https://docs.databricks.com/(https://docs.databricks.com/aws/generative-ai/agent-evaluation) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-evaluation/)).

To evaluate your tool calls, add custom metrics. See Databricks documentation ([AWS](https://docs.databricks.com/en/generative-ai/agent-evaluation/custom-metrics.html#evaluating-tool-calls) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-evaluation/custom-metrics#evaluating-tool-calls)).

In [0]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, RetrievalGroundedness, RetrievalRelevance, Safety

eval_dataset = [
    {
        "inputs": {"input": [{"role": "user", "content": "Calculate the 15th Fibonacci number"}]},
        "expected_response": "The 15th Fibonacci number is 610.",
    }
]

eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: AGENT.predict({"input": input}),
    scorers=[RelevanceToQuery(), Safety()],  # add more scorers here if they're applicable
)

# Review the evaluation results in the MLfLow UI (see console output)

2026/05/19 02:13:02 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/05/19 02:13:03 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/1 [Elapsed: 00:00, Remaining: ?]

# Pre-deployment agent validation
Before registering and deploying the agent, perform pre-deployment checks using the mlflow.models.predict() API.

In [0]:
import json
import mlflow

# Load the model ONCE — avoids re-loading for every predict call
loaded_model = mlflow.pyfunc.load_model(f"runs:/{logged_agent_info.run_id}/agent")

# Use a consistent thread_id to test short-term memory across calls
test_thread_id = "pre-deploy-validation-thread-v2"

# Turn 1: Establish context
print("=== Turn 1: Establish context ===")
result1 = loaded_model.predict(
    {
        "input": [{"role": "user", "content": "My name is X and I'm debugging a pipeline failure in engen_prod.silver_member.member_dim"}],
        "custom_inputs": {"thread_id": test_thread_id},
    }
)
print(json.dumps(result1, indent=2, default=str))

# Turn 2: Ask a follow-up that requires memory of Turn 1
print("\n=== Turn 2: Test memory (same thread) ===")
result2 = loaded_model.predict(
    {
        "input": [{"role": "user", "content": "What table was I asking about, and what's my name?"}],
        "custom_inputs": {"thread_id": test_thread_id},
    }
)
print(json.dumps(result2, indent=2, default=str))

# Turn 3: New thread — should NOT remember anything
print("\n=== Turn 3: No memory (different thread) ===")
result3 = loaded_model.predict(
    {
        "input": [{"role": "user", "content": "What table was I asking about, and what's my name?"}],
        "custom_inputs": {"thread_id": "different-thread-no-context"},
    }
)
print(json.dumps(result3, indent=2, default=str))

=== Turn 1: Establish context ===
{
  "object": "response",
  "output": [
    {
      "type": "message",
      "id": "lc_run--019e3e0a-bd3b-7193-afb5-ac53bafff75b",
      "content": [
        {
          "text": "Hi Ananya! Happy to help you debug the pipeline failure in `engen_prod.silver_member.member_dim`.\n\nTo get started with troubleshooting, could you provide me with:\n\n1. **Job run ID** \u2013 so I can help trace the specific execution.\n2. **Error message** \u2013 any error output or stack trace you're seeing.\n\nAny additional context (e.g., when it started failing, whether there were recent schema or upstream changes) would also be helpful!",
          "type": "output_text",
          "annotations": []
        }
      ],
      "role": "assistant"
    }
  ],
  "custom_outputs": {
    "thread_id": "pre-deploy-validation-thread-v2"
  }
}

=== Turn 2: Test memory (same thread) ===
{
  "object": "response",
  "output": [
    {
      "type": "message",
      "id": "lc_run--019e3e

[Trace(trace_id=tr-aebc14fab3c4dd3ed156626263cbc3d8), Trace(trace_id=tr-f5eea8faf480f7a640338f18239779f3), Trace(trace_id=tr-93c0b41452e62cf685a9579b41a8ea2d)]

# Register the model to Unity Catalog
Update the `catalog`, `schema`, and `model_name` below to register the MLflow model to Unity Catalog.

In [0]:
mlflow.set_registry_uri("databricks-uc")

# TODO: define the catalog, schema, and model name for your UC model
catalog = "catalog"
schema = "schema"
model_name = "short-term-memory-agent"

UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME
)

Successfully registered model 'ananyaroy.lakebase-simple.short-term-memory-agent'.


Uploading artifacts:   0%|          | 0/13 [00:00<?, ?it/s]

🔗 Created version '1' of model 'ananyaroy.lakebase-simple.short-term-memory-agent': https://adb-984752964297111.11.azuredatabricks.net/explore/data/models/ananyaroy/lakebase-simple/short-term-memory-agent/version/1?o=984752964297111


Deploy the agent

In [0]:
from databricks import agents
agents.deploy(UC_MODEL_NAME, uc_registered_model_info.version, tags = {"endpointSource": "docs"}, deploy_feedback_model=False)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-a7b1767d-4c47-4239-bad4-70c3e2c712f0/lib/python3.10/site-packages/databricks/agents/deployments.py:641: UserWarning: This endpoint is being deployed without a feedback model, which has been deprecated.
For more information, see: https://docs.databricks.com/aws/en/generative-ai/agent-framework/feedback-model
  warnings.warn(



    Deployment of ananyaroy.lakebase-simple.short-term-memory-agent version 1 initiated.  This can take up to 15 minutes and the Review App & Query Endpoint will not work until this deployment finishes.

    View status: https://adb-984752964297111.11.azuredatabricks.net/ml/endpoints/agents_ananyaroy-lakebase-simple-short-term-memory-agent/?o=984752964297111
    Review App: https://adb-984752964297111.11.azuredatabricks.net/ml/review-v2/0b5a356df96a421cb1b830c05e53bdb2/chat?o=984752964297111

You can refer back to the links above from the endpoint detail page at https://adb-984752964297111.11.azuredatabricks.net/ml/endpoints/agents_ananyaroy-lakebase-simple-short-term-memory-agent/?o=984752964297111.

To set up monitoring for your deployed agent, see:
https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/production-monitoring


Deployment(model_name='ananyaroy.lakebase-simple.short-term-memory-agent', model_version='1', endpoint_name='agents_ananyaroy-lakebase-simple-short-term-memory-agent', served_entity_name='ananyaroy-lakebase-simple-short-term-memory-agent_1', query_endpoint='https://adb-984752964297111.11.azuredatabricks.net/serving-endpoints/agents_ananyaroy-lakebase-simple-short-term-memory-agent/served-models/ananyaroy-lakebase-simple-short-term-memory-agent_1/invocations?o=984752964297111', endpoint_url='https://adb-984752964297111.11.azuredatabricks.net/ml/endpoints/agents_ananyaroy-lakebase-simple-short-term-memory-agent/?o=984752964297111', review_app_url='https://adb-984752964297111.11.azuredatabricks.net/ml/review-v2/0b5a356df96a421cb1b830c05e53bdb2/chat?o=984752964297111')

# Next steps
It will take around 15 minutes for you to finish deploying your agent. After your agent is deployed, you can chat with it in Review App/playground to perform additional checks, share it with SMEs in your organization for feedback, or embed it in a production application. 

Query your Lakebase instance to see a record of your conversation at various threads/checkpoints. Here is a basic query to see the 10 most recent checkpoints:

```
SELECT
    c.*,
    (c.checkpoint::json->>'ts')::timestamptz AS ts
FROM checkpoints c
ORDER BY ts DESC
LIMIT 10;
```